# Download ConceptGraphs source

Clone the [official repository](https://github.com/concept-graphs/concept-graphs) into `dependencies/concept-graphs` for code analysis. Requires Git and network access. Run from the project root or the `code` folder. This downloads source only; model weights, datasets, and runtime packages are separate setup steps.

Before downloading, list the contents of `dependencies`. Reuse an existing ConceptGraphs checkout only after checking its origin, commit, Git object integrity, and every tracked file. Existing files are preserved; incomplete checkouts are reported instead of overwritten. Other dependency folders are listed, not assumed to be valid installations. The printed commit identifies the version being studied.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tempfile
import os
import re
import time
import threading
import queue
import html
from IPython.display import HTML, display

def clone_with_progress(url, destination, idle_timeout=120, total_timeout=600):
    # Git reports separate percentages for receiving, resolving and checkout.
    panel = display(HTML('<progress></progress> Connecting to GitHub...'), display_id=True)
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    command = ['git', '-c', 'http.lowSpeedLimit=1024', '-c', 'http.lowSpeedTime=60', 'clone', '--depth', '1', '--single-branch', '--progress', url, str(destination)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=env)
    messages = queue.Queue()
    def read_output():
        try:
            for line in process.stdout:
                messages.put(line.strip())
        finally:
            messages.put(None)
    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()
    start = last_activity = time.monotonic()
    recent = []
    status = 'Connecting to GitHub...'
    percent = None
    try:
        while True:
            try:
                line = messages.get(timeout=0.5)
            except queue.Empty:
                line = ''
            if line is None:
                break
            now = time.monotonic()
            if line:
                last_activity = now
                recent = (recent + [line])[-15:]
                status = line
                match = re.search(r'(\d+)%', line)
                percent = int(match.group(1)) if match else None
            value = '' if percent is None else f' value={percent} max=100'
            panel.update(HTML(f'<progress{value}></progress> {html.escape(status)}<br>Elapsed: {now-start:.0f}s; no new output: {now-last_activity:.0f}s'))
            if now - last_activity > idle_timeout or now - start > total_timeout:
                raise TimeoutError('Download stalled or exceeded 10 minutes. Check GitHub connectivity before retrying.')
        if process.wait(timeout=10):
            raise RuntimeError('Git download failed:' + chr(10) + chr(10).join(recent))
        panel.update(HTML('<progress value=100 max=100></progress> Download complete; verifying files...'))
    except BaseException:
        if process.poll() is None:
            if os.name == 'nt':
                subprocess.run(['taskkill', '/PID', str(process.pid), '/T', '/F'], capture_output=True)
            else:
                process.kill()
            process.wait(timeout=10)
        panel.update(HTML('<b>Download stopped.</b> See the error below.'))
        raise
    finally:
        reader.join(timeout=2)
        process.stdout.close()

def run_git(*args):
    result = subprocess.run(['git', *map(str, args)], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f'Git failed (exit {result.returncode}): {result.stderr.strip() or result.stdout.strip()}')
    return result.stdout.strip()

def verify_checkout(path):
    run_git('-C', path, 'rev-parse', '--verify', 'HEAD^{commit}')
    run_git('-C', path, 'fsck', '--connectivity-only', '--no-dangling')
    tracked = run_git('-C', path, 'ls-tree', '-r', '--name-only', '-z', 'HEAD').split(chr(0))
    missing = [name for name in tracked if name and not (path / name).exists()]
    if missing:
        raise RuntimeError(f'{len(missing)} tracked files are missing: {missing[:10]}')
    changes = run_git('-C', path, 'status', '--porcelain')
    if changes:
        print('Local changes detected and preserved:' + chr(10) + changes)
    print(f'Verified {sum(bool(name) for name in tracked)} tracked files.')

if shutil.which('git') is None:
    raise RuntimeError('Install Git and restart the notebook kernel first.')

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents) if (p / 'code' / 'download_dependencies.ipynb').is_file()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the capstone project folder or its code folder.')
repo_url = 'https://github.com/concept-graphs/concept-graphs.git'
repo_path = project_root / 'dependencies' / 'concept-graphs'
repo_path.parent.mkdir(parents=True, exist_ok=True)
print(f'Dependencies folder: {repo_path.parent}')
entries = sorted(repo_path.parent.iterdir(), key=lambda p: p.name.lower())
if not entries:
    print('No dependencies downloaded yet.')
for entry in entries:
    kind = 'folder' if entry.is_dir() else 'file'
    note = ' (ConceptGraphs: will verify)' if entry == repo_path else ' (unmanaged: preserved, not validated)'
    print(f'  [{kind}] {entry.name}{note}')

if not repo_path.exists():
    # Publish the final folder only after a successful, verified clone.
    staging = Path(tempfile.mkdtemp(prefix='concept-graphs-download-', dir=repo_path.parent))
    try:
        staged_repo = Path(staging) / 'repo'
        print('Downloading ConceptGraphs; wait for the completion message.')
        clone_with_progress(repo_url, staged_repo)
        verify_checkout(staged_repo)
        staged_repo.rename(repo_path)
        staging.rmdir()
    except BaseException:
        print(f'Incomplete download preserved at {staging}; the next run uses a fresh staging folder.')
        raise
elif not (repo_path / '.git').exists():
    raise RuntimeError(f'{repo_path} exists but is not a Git checkout. Inspect it before retrying.')
else:
    origin = run_git('-C', repo_path, 'remote', 'get-url', 'origin')
    if origin != repo_url:
        raise RuntimeError(f'Unexpected repository origin: {origin}')
    try:
        verify_checkout(repo_path)
    except RuntimeError as exc:
        raise RuntimeError(f'Checkout is incomplete at {repo_path}. Wait for any active download to finish. If it has failed, rename this folder to preserve it, then rerun this cell. Details: {exc}') from exc
    print('Using existing checkout; no files changed.')

print(f'Source: {repo_path}')
for args in (['branch', '--show-current'], ['rev-parse', 'HEAD']):
    print(run_git('-C', repo_path, *args))
print('ConceptGraphs source is ready.')


## Download the OpenEQA HM3D and ScanNet scene histories

Download the per-episode archives from the `AIGeeksGroup/OpenEQA` Hugging Face mirror. Extracted scenes go to `dataset rgb only/frames/hm3d-v0` and `dataset rgb only/frames/scannet-v0`. Downloads are cached under `dataset rgb only/_downloads`. Existing non-empty episode folders are skipped, so the loop can be rerun after an interruption.

Install the downloader once if necessary with `python -m pip install huggingface_hub`.

In [ ]:
import tarfile
from pathlib import Path

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError as exc:
    raise RuntimeError(
        'Install the downloader first: python -m pip install huggingface_hub'
    ) from exc

RGB_REPO_ID = 'AIGeeksGroup/OpenEQA'
RGB_SPLITS = ('hm3d-v0', 'scannet-v0')
cwd = Path.cwd().resolve()
project_root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / 'code' / 'download_dependencies.ipynb').is_file()),
    None,
)
if project_root is None:
    raise RuntimeError(
        'Run this notebook from the capstone project folder or its code folder.'
    )
RGB_ROOT = project_root / 'dataset rgb only'
RGB_FRAMES_ROOT = RGB_ROOT / 'frames'
RGB_DOWNLOAD_ROOT = RGB_ROOT / '_downloads'

repo_files = HfApi().list_repo_files(RGB_REPO_ID, repo_type='dataset')

for split in RGB_SPLITS:
    archives = sorted(
        name for name in repo_files
        if name.startswith(f'{split}/') and name.endswith('.tar')
    )
    if not archives:
        raise RuntimeError(f'No .tar episodes found for {split} in {RGB_REPO_ID}.')

    split_frames = RGB_FRAMES_ROOT / split
    split_frames.mkdir(parents=True, exist_ok=True)
    print(f'\n{split}: {len(archives)} episodes')

    for number, repo_file in enumerate(archives, 1):
        episode = Path(repo_file).stem
        destination = split_frames / episode
        if destination.is_dir() and any(destination.iterdir()):
            print(f'[{number}/{len(archives)}] {episode}: already extracted')
            continue

        print(f'[{number}/{len(archives)}] {episode}: downloading')
        archive = Path(hf_hub_download(
            repo_id=RGB_REPO_ID,
            repo_type='dataset',
            filename=repo_file,
            local_dir=RGB_DOWNLOAD_ROOT,
        ))

        staging = split_frames / f'.extracting-{episode}'
        if staging.exists():
            raise RuntimeError(
                f'Incomplete extraction exists at {staging}. Inspect or remove it, then rerun.'
            )
        staging.mkdir()
        try:
            with tarfile.open(archive, 'r') as bundle:
                bundle.extractall(staging, filter='data')
            extracted = staging / episode
            if not extracted.is_dir() or not any(extracted.iterdir()):
                raise RuntimeError(f'{archive.name} did not contain {episode}/.')
            if destination.exists():
                raise RuntimeError(f'Empty destination already exists: {destination}')
            extracted.rename(destination)
            staging.rmdir()
        except BaseException:
            print(f'Extraction stopped; temporary files remain at {staging}.')
            raise

        print(f'[{number}/{len(archives)}] {episode}: ready')

print('\nHM3D and ScanNet scene-history download loop finished.')
